# Draw Your Own Digit Demo

This is the planned live PYNQ-Z2 demo: draw a digit, preprocess it into MNIST format, run inference, and display the prediction. Today it can run the software reference path. After the AXI wrapper exists, the same flow can call the hardware driver.

Install once on the PYNQ image if needed:

```bash
pip install ipycanvas pillow
```

In [ ]:
import sys
from pathlib import Path

repo = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(repo / 'Software' / 'mnist'))

import numpy as np
import matplotlib.pyplot as plt

from draw_digit import canvas_rgba_to_grayscale, preprocess_drawn_digit, ascii_preview
from pynq_demo import software_tile_reference, AcceleratorDriver


In [ ]:
artifact = repo / 'artifacts' / 'mnist' / 'mnist_softmax_int8.npz'
data = np.load(artifact)
weights = data['weights_int8']
bias = data['bias_int8']

# Use software-only until the AXI wrapper and bitstream are ready.
USE_HARDWARE = False
driver = AcceleratorDriver(None)


In [ ]:
from ipycanvas import Canvas, hold_canvas
from ipywidgets import Button, HBox, VBox, Output

canvas_size = 280
canvas = Canvas(width=canvas_size, height=canvas_size, sync_image_data=True)
canvas.layout.border = '2px solid #333'

drawing = {'active': False, 'last': None}
out = Output()

def clear_canvas(*_):
    with hold_canvas(canvas):
        canvas.fill_style = 'white'
        canvas.fill_rect(0, 0, canvas_size, canvas_size)
        canvas.stroke_style = 'black'
        canvas.line_width = 18
        canvas.line_cap = 'round'
        canvas.line_join = 'round'
    out.clear_output()

def on_down(x, y):
    drawing['active'] = True
    drawing['last'] = (x, y)

def on_move(x, y):
    if not drawing['active'] or drawing['last'] is None:
        return
    lx, ly = drawing['last']
    with hold_canvas(canvas):
        canvas.stroke_line(lx, ly, x, y)
    drawing['last'] = (x, y)

def on_up(x, y):
    drawing['active'] = False
    drawing['last'] = None

canvas.on_mouse_down(on_down)
canvas.on_mouse_move(on_move)
canvas.on_mouse_up(on_up)

def run_inference(*_):
    rgba = np.asarray(canvas.get_image_data(0, 0, canvas_size, canvas_size).data)
    rgba = rgba.reshape(canvas_size, canvas_size, 4)
    gray = canvas_rgba_to_grayscale(rgba)
    drawn = preprocess_drawn_digit(gray, canvas_size=canvas_size)

    if USE_HARDWARE:
        logits = driver.run_image(drawn.packed_words)
    else:
        logits, per_tile = software_tile_reference(drawn.image_int8, weights, bias)

    pred = int(np.argmax(logits))
    driver.write_led_digit(pred)

    out.clear_output()
    with out:
        print(f'Predicted digit: {pred}')
        print(f'LED mirror: {pred:04b}')
        print('Logits:', logits.tolist())
        print('28x28 preview:')
        print(ascii_preview(drawn.image_int8))
        plt.figure(figsize=(2, 2))
        plt.imshow(drawn.image_float, cmap='gray')
        plt.title(f'Prediction: {pred}')
        plt.axis('off')
        plt.show()

clear_btn = Button(description='Clear')
run_btn = Button(description='Run inference', button_style='success')
clear_btn.on_click(clear_canvas)
run_btn.on_click(run_inference)

clear_canvas()
VBox([canvas, HBox([clear_btn, run_btn]), out])
